# Polymarket vs Kalshi: activity and last-print gaps by competition

This notebook studies the full population's trading activity, then price gaps, fee-adjusted last-print screens, frequency and episode duration in candidate competition-season × phase cells. A candidate uses the smallest F in 5 / 10 / 60 seconds reaching 70% dual-fresh coverage (§2c). Activity diagnostics run before this filter.

**Contents:** §1 population; §2 availability and directional trade waiting; §3–5 signed price gaps; §6 fee-adjusted last-print screens; §7 conditions; §8 frequency; §9 event-time episodes. Subsequent price-response, passive-fill and basket research is in [02.2](02.2_cross_venue_confirmation_and_response.ipynb).

**Inputs:** `availability.csv`, `trade_waits.csv` from [`availability.py`](../availability.py); `gap_cents.csv` from [`gap_cents.py`](../gap_cents.py); `matches`, `audit`, `screen`, `scenarios`, `conditions`, `episodes` from `gaps.py`. The screen uses each candidate's F; scenarios and conditions exist only at strict 5s. Shared candidate selection and chart helpers live in [`competition.py`](../competition.py).

**Vocabulary.** A print aggregates a venue's fills within one second, outcome and taker side into a volume-weighted price; the pooled `all` leg merges both sides. A pair combines the venues' latest prints. Its age is the older print's age; both are fresh when age ≤ F. An observation is a print on either pair leg while the other leg is fresh. A route assigns venue roles and which venue holds YES. F looks backward; the activity waiting horizon H looks forward.

Trade prices are references, not executable quotes. The tape has no books, depth, queues or fills, and these screens do not establish P&L.

In [ ]:
from pathlib import Path
import subprocess, sys
from itertools import product
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'research/soccer_1x2_analysis/ticks.py').exists())
ANALYSIS = ROOT / 'research/soccer_1x2_analysis'
sys.path.insert(0, str(ANALYSIS))
import gaps, ticks

OUT = ANALYSIS / 'outputs/gap_analysis'
CFG = gaps.Config()
STRICT = CFG.freshness_sec
PHASES = list(gaps.PHASES)
PHASE_SEC = gaps.phase_seconds(CFG)
def ensure(csv, script, *args):
    if not (OUT / csv).exists():
        subprocess.run([sys.executable, str(ANALYSIS / script), *args], check=True)
    return pd.read_csv(OUT / csv)
avail = ensure('availability.csv', 'availability.py', '--max-f', '300')
if 'neither_pct_of_clock' not in avail or not (OUT / 'trade_waits.csv').exists():
    subprocess.run([sys.executable, str(ANALYSIS / 'availability.py'), '--max-f', '300'], check=True)
    avail = pd.read_csv(OUT / 'availability.csv')
waits = pd.read_csv(OUT / 'trade_waits.csv')
cents = ensure('gap_cents.csv', 'gap_cents.py', '--freshness', '5', '10', '60')
m = gaps.read_export(OUT / 'matches.parquet')   # ticks.matches() plus cohort and complete_market_set
m['league'] = m.league.astype(str)
m['kickoff_iso'] = pd.to_datetime(m.kickoff_iso.astype(str), utc=True)
m['combo'] = m.league + ' ' + m.season.astype(str) + '/' + (m.season % 100 + 1).astype(str).str.zfill(2)
COMBO = m.set_index('match_id').combo
for df in (avail, cents, waits):
    df['combo'] = df.league + ' ' + df.season.astype(str) + '/' + (df.season % 100 + 1).astype(str).str.zfill(2)
FIXTURES = m.groupby('combo').size()
def export(name, columns=None):
    df = gaps.read_export(OUT / f'{name}.parquet', columns)
    df['combo'] = df.match_id.map(COMBO)
    return df
audit = export('audit')
plt.rcParams.update({'figure.dpi': 100, 'axes.grid': True, 'grid.alpha': .3, 'axes.titlesize': 9, 'font.size': 9})
from competition import BLUE, RED, GREEN, PURPLE, grid, finish, bars, phase_coverage, select_candidates
print(f'{len(m):,} fixtures, {m.combo.nunique()} competition-seasons; phases {PHASES}')

## 1. Population by competition and season

Same universe as 02 (linked on both venues, grounded to API-Football), broken out by `season` (API-Football season start year; 2025 = 2025/26). Trades per fixture is the number to keep in mind for everything below: the World Cup is 18× the Premier League and 66× Ligue 1.

In [ ]:
cov = m.merge(audit[['match_id', 'rows', 'pm_rows', 'k_rows', 'window_rows']], on='match_id')
cov['both_traded'] = (cov.pm_rows > 0) & (cov.k_rows > 0)
pop = cov.groupby(['combo'], observed=True).agg(fixtures=('match_id', 'size'), both_traded=('both_traded', 'sum'),
                                              complete_1x2=('complete_market_set', 'sum'), trades=('rows', 'sum'), pm_trades=('pm_rows', 'sum'), k_trades=('k_rows', 'sum'),
                                              first_kickoff=('kickoff_iso', 'min'), last_kickoff=('kickoff_iso', 'max'))
for c in ('trades', 'pm_trades', 'k_trades'):
    pop[f'{c}_per_fixture'] = (pop[c] / pop.fixtures).round(0).astype(int)
pop = pop.sort_values('trades_per_fixture', ascending=False)
display(pop)
fig, ax = plt.subplots(figsize=(12, 3.6))
x = np.arange(len(pop))
ax.bar(x - .2, pop.pm_trades_per_fixture, .4, color=BLUE, label='Polymarket'); ax.bar(x + .2, pop.k_trades_per_fixture, .4, color=RED, label='Kalshi')
ax.set_yscale('log'); ax.set_xticks(x); ax.set_xticklabels(pop.index, rotation=30, ha='right', fontsize=8); ax.set_ylabel('trades per fixture (log)')
ax.legend(); ax.set_title('Trades per fixture by competition-season'); plt.tight_layout(); plt.show()

## 2. Availability and directional trade waiting — full population

Freshness F looks backward: a venue is fresh when its latest print is at most F seconds old. Forward horizon H asks whether the other venue prints strictly after the trigger and within H seconds. These answer different questions and use different denominators.

Availability uses the entire phase clock for all three outcomes in every fixture, including silent outcomes. Duration intervals are split at phase boundaries. The in-play split at 55 minutes is a clock-based diagnostic, not an observed halftime boundary. All activity diagnostics below run before candidate selection; low dual freshness does not by itself rule out an asynchronous strategy.

In [ ]:
PH5 = ['pre_24h_to_1h', 'pre_last_hour', 'post_0_to_55m', 'post_55_to_105m', 'post_105_to_135m']
colors5 = dict(zip(PH5, [BLUE, RED, GREEN, '#19a974', PURPLE]))
combos = FIXTURES.sort_index()
fig, axes = grid(2, 5, w=4.2, h=3.6, sharex=True, sharey=True)
for ax, (combo, n) in zip(axes.flat, combos.items()):
    d = avail[avail.combo == combo]
    for ph in PH5:
        q = d[d.phase == ph].sort_values('freshness')
        ax.plot(q.freshness, q.fresh_pct_of_clock, color=colors5[ph], lw=1.8, label=ph)
    ax.set_title(f'{combo}  (n={n})'); ax.set_ylim(0, 100)
for ax in axes[-1]: ax.set_xlabel('Freshness F (s)')
for ax in axes[:, 0]: ax.set_ylabel('% of outcome-seconds with both venues fresh')
axes.flat[0].legend(fontsize=8, loc='upper left')
fig.suptitle('How much of the clock has a fresh cross-venue pair?', fontsize=13); fig.tight_layout(); plt.show()

# merge the two in-play halves (weighted by length) into the pipeline's post_0_to_105m
w = {'post_0_to_55m': 55 * 60, 'post_55_to_105m': 50 * 60}
cov4 = phase_coverage(avail)
h60 = avail[(avail.freshness == 60) & avail.phase.isin(w)].pivot(index='combo', columns='phase', values='fresh_pct_of_clock')
print(f'largest |second - first half| coverage difference at F=60: {(h60.post_55_to_105m - h60.post_0_to_55m).abs().max():.1f} pp -> keep post_0_to_105m merged')



### 2a. Where does the rest of the clock go?

Four mutually exclusive states sum to 100%: both fresh, only Kalshi fresh, only PM fresh, neither fresh. “Neither fresh” means neither venue traded in the preceding F seconds; it does not mean the data are missing. Conversely, a second without a new trade can still have both venues fresh. A missing tape file stops computation rather than being classified as inactivity.

Denominator: fixture × outcome × phase seconds. The plots retain all competition-seasons, at common F = 5 / 10 / 60 seconds, before applying the candidate rule. Prints before the analysis start are unavailable, so initial freshness and “no prior print” refer to the observed analysis window.

In [ ]:
STATE_COLS = ['fresh_pct_of_clock', 'k_only_pct_of_clock', 'pm_only_pct_of_clock', 'neither_pct_of_clock']
STATE_NAMES = ['Both fresh', 'Only Kalshi fresh', 'Only PM fresh', 'Neither fresh']
state_half = avail[avail.phase.isin(w)].assign(wt=lambda d: d.phase.map(w))
state_merged = (state_half.assign(**{col: state_half[col] * state_half.wt for col in STATE_COLS})
    .groupby(['combo', 'freshness'])[STATE_COLS + ['wt']].sum())
state_merged[STATE_COLS] = state_merged[STATE_COLS].div(state_merged.wt, axis=0)
state4 = pd.concat([avail[~avail.phase.isin(w)][['combo', 'phase', 'freshness'] + STATE_COLS],
    state_merged.reset_index().assign(phase='post_0_to_105m')[['combo', 'phase', 'freshness'] + STATE_COLS]], ignore_index=True)
assert np.allclose(state4[STATE_COLS].sum(axis=1), 100, atol=1e-6)
for F in (5, 10, 60):
    fig, axes = grid(2, 2, w=9, h=4.7, sharex=True)
    for ax, ph in zip(axes.flat, PHASES):
        d = state4[(state4.phase == ph) & (state4.freshness == F)].set_index('combo').reindex(combos.index)
        left = np.zeros(len(d))
        for col, label, color in zip(STATE_COLS, STATE_NAMES, [GREEN, RED, BLUE, '#bdbdbd']):
            ax.barh(d.index, d[col], left=left, color=color, label=label)
            left += d[col].to_numpy()
        ax.set_title(ph); ax.set_xlim(0, 100); ax.set_xlabel('% of all outcome-seconds'); ax.invert_yaxis()
    axes.flat[0].legend(fontsize=7, loc='lower right')
    fig.suptitle(f'Full-clock activity states: F={F}s'); fig.tight_layout(); plt.show()


### 2b. After a trade, how soon does the other venue trade?

For each fixture and outcome, each source-venue trade second counts once, merging buy/sell fills. The denominator includes **all** source trade seconds, without requiring an existing fresh pair. The numerator is a destination print in **(t, t+H]**. Same-second destination prints are reported separately and do not count as a forward response; those source triggers remain in the denominator.

H = 1 / 5 / 10 / 30 / 60 / 300 seconds. Each trigger must have its entire H-second future inside its phase and the recorded analysis window; boundary-censored triggers are reported and excluded from the rate. The two in-play subphases are preserved for this check, even when pooled into the 0–105 minute reporting phase. Thus denominators vary with H and the displayed curves need not be strictly monotone.

The table also splits triggers by destination state at t: fresh (age ≤ F), stale (a previous observed print exists, age > F), or no prior observed print. F and H are independent. Rates are pooled over source trade seconds, not averaged equally across fixtures. Zero eligible triggers give an undefined rate, not zero.

These are descriptive waiting probabilities, not evidence that the source causes or predicts destination activity above its baseline. That requires controls for market, time and activity, and uncertainty clustered by fixture.

In [ ]:
wait4 = waits.copy()
wait4['phase'] = wait4.phase.replace({'post_0_to_55m': 'post_0_to_105m', 'post_55_to_105m': 'post_0_to_105m'})
WAIT_COUNTS = ['triggers', 'same_second', 'eligible', 'censored', 'followed']
wg = wait4.groupby(['combo', 'phase', 'source', 'freshness', 'other_state', 'horizon'], observed=True)[WAIT_COUNTS].sum().reset_index()
wg['follow_pct'] = 100 * wg.followed / wg.eligible.replace(0, np.nan)
assert (wg.followed <= wg.eligible).all()
assert (wg.triggers == wg.eligible + wg.censored).all()
# The all-trigger population is independent of F; use F=60 once, without tripling it.
for ph in PHASES:
    fig, axes = grid(2, 5, w=4.2, h=3.2, sharex=True, sharey=True)
    for ax, combo in zip(axes.flat, combos.index):
        for src, color, label in [('KALSHI', RED, 'Kalshi → PM'), ('POLYMARKET', BLUE, 'PM → Kalshi')]:
            d = wg[(wg.combo == combo) & (wg.phase == ph) & (wg.source == src) & (wg.other_state == 'all') & (wg.freshness == 60)].sort_values('horizon')
            ax.plot(d.horizon, d.follow_pct, marker='o', ms=3, color=color, label=label)
        ax.set_title(combo); ax.set_xscale('log'); ax.set_ylim(0, 100)
    for ax in axes[-1]: ax.set_xlabel('Forward horizon H (s)')
    for ax in axes[:, 0]: ax.set_ylabel('% with a strictly later destination trade')
    axes.flat[0].legend(fontsize=7)
    fig.suptitle(f'All source trade seconds: {ph}'); fig.tight_layout(); plt.show()

# Full-population state comparison, adjustable independently from the curves.
DETAIL_F, DETAIL_H = 60, 60
wd = wg[(wg.freshness == DETAIL_F) & (wg.horizon == DETAIL_H)].copy()
wd['same_second_pct'] = 100 * wd.same_second / wd.triggers.replace(0, np.nan)
wd['censored_pct'] = 100 * wd.censored / wd.triggers.replace(0, np.nan)
print(f'Destination state at trigger: F={DETAIL_F}s; forward waiting horizon H={DETAIL_H}s')
with pd.option_context('display.max_rows', None):
    display(wd.pivot(index=['combo', 'phase', 'source'], columns='other_state', values='follow_pct').round(1))
print('All-trigger denominators and same-second overlap (before boundary censoring):')
with pd.option_context('display.max_rows', None):
    display(wd[wd.other_state == 'all'].set_index(['combo', 'phase', 'source'])[
        ['triggers', 'eligible', 'followed', 'same_second_pct', 'censored_pct']].round(1))


**Reading the current export (2025/26, in play).** At F=60s, the Premier League clock is 78.31% both fresh, 15.68% only Kalshi fresh, 2.63% only PM fresh and 3.38% neither fresh. The Champions League split is 69.68% / 18.70% / 4.43% / 7.19%. Thus most of the excluded clock is one-sided freshness, especially Kalshi fresh while PM is stale, rather than both venues being stale.

With H=60s, after **all** Kalshi trade seconds PM prints again in time in 93.6% of eligible Premier League triggers and 93.5% of Champions League triggers. When PM is already stale at the trigger (F=60s), those rates are only 61.8% and 60.0%. In the reverse direction, all-trigger rates are 98.8% and 97.2%, but fall to 70.1% and 36.0% when Kalshi is stale. These are event-weighted rates with phase-boundary censoring; they cannot be inferred from the clock-weighted availability percentages.

The existing fresh-only analysis therefore describes a materially different activity population. This motivates retaining the stale-trigger groups for subsequent price-response research; it does not establish causal leadership or profitability.

### 2c. Candidate cells for the fresh-pair gap study

A competition-season × phase qualifies if dual freshness at F=60s covers at least 70% of its clock (rounded to a whole percent). Choose the smallest F in 5 / 10 / 60s reaching that threshold. This is a scope rule for the following fresh-pair price comparisons, not a rejection of asynchronous opportunities elsewhere. The full-population diagnostics above remain independent of this selection.

In [ ]:
SCREEN_GRID = sorted(set(gaps.read_export(OUT / 'screen.parquet', ['freshness']).freshness.astype(int)))
table, CANDS, F_PIPE = select_candidates(cov4, FIXTURES, PHASES, SCREEN_GRID)
LABEL = {c: f'{c[0]}' + chr(10) + f'{c[1]}  F={c[2]}s' for c in CANDS}
display(table)
print('pipeline F used:', {LABEL[c].replace(chr(10), ' | '): F_PIPE[c] for c in CANDS})

## 3. The state, freshness, and the signed gap

As in 02 §2: one row per second in which either venue printed, carrying each leg's last print; an **observation** is a print on either leg while the other leg is ≤ F old (event weighting, the taker's view); the same print weighted by the seconds its pair then stays fresh, `clip(F − age, 0, dur)`, gives the time weighting (the resting maker's view). Bars below are the share of fresh prints by pooled PM − K in cents: the centre bucket (−1, 1) holds the sub-cent noise, `[1, 2)` / `(−2, −1]` hold the exact one-cent gaps both venues quote, and the ends are open at ±4 c; lighter bars are time-weighted. Means in the tables come from exact sums, not bucket midpoints. Columns: candidate cells; rows: outcome selection (`team_win` = home + away, priced separately, pooled only for reporting).

In [ ]:
OUTCOME_SEL = {'all': ['home', 'draw', 'away'], 'team_win': ['home', 'away'], 'draw': ['draw']}
C = list(gaps.GAP_CENTS)
cent_labels = {-4: '≤−4', -3: '(−4,−3]', -2: '(−3,−2]', -1: '(−2,−1]', 0: '(−1,1)', 1: '[1,2)', 2: '[2,3)', 3: '[3,4)', 4: '≥4'}
cent_labels = [cent_labels[c] for c in C]
def cell(frame, c, fcol='freshness'):
    combo, phase, F = c
    return frame[(frame.combo == combo) & (frame.phase == phase) & (frame[fcol] == F)]
def cent_stats(frame):
    g = frame.groupby('cents')[['n', 'time', 'gap_sum', 'abs_sum', 'gap_time_sum']].sum().reindex(C, fill_value=0)
    n, c = g.n.sum(), np.array(C)
    pct = 100 * g[['n', 'time']] / g[['n', 'time']].sum().replace(0, np.nan)
    return pct, dict(prints=int(n), mean_gap_c=g.gap_sum.sum() / max(n, 1), mean_abs_gap_c=g.abs_sum.sum() / max(n, 1),
                     ge_2c_pct=100 * g.n[np.abs(c) >= 2].sum() / max(n, 1),
                     pm_lower_1c_pct=100 * g.n[c <= -1].sum() / max(n, 1), pm_higher_1c_pct=100 * g.n[c >= 1].sum() / max(n, 1),
                     time_mean_gap_c=g.gap_time_sum.sum() / max(g.time.sum(), 1e-9))
summary = []
fig, axes = grid(len(OUTCOME_SEL), len(CANDS), sharex=True, sharey=True)
for i, (name, outcomes) in enumerate(OUTCOME_SEL.items()):
    for j, c in enumerate(CANDS):
        d = cell(cents, c); d = d[(d.pair == 'all/all') & d.outcome.isin(outcomes)]
        pct, st = cent_stats(d)
        bars(axes[i, j], cent_labels, {'per print': pct.n, 'per second': pct.time}, [BLUE, '#b3b8ff'])
        axes[i, j].set_title(f'n={st["prints"]:,}', fontsize=8)
        summary.append(dict(outcome=name, cell=LABEL[c].replace(chr(10), ' | '), **st))
for ax in axes[-1]: ax.set_xlabel('PM − K (cents)'); ax.tick_params(axis='x', rotation=45)
axes[0, 0].legend(fontsize=8)
finish(fig, axes, list(OUTCOME_SEL), [LABEL[c] for c in CANDS], '% of fresh prints', 'Signed gap distribution per candidate cell')
summary = pd.DataFrame(summary)
display(summary.pivot_table(index='cell', columns='outcome', values=['mean_gap_c', 'mean_abs_gap_c', 'ge_2c_pct', 'pm_lower_1c_pct', 'pm_higher_1c_pct'])
        .reindex(columns=['all', 'team_win', 'draw'], level=1).round(2))

## 4. Action pairs: the signed gap is a route's gross edge

Signed mean PM − K per fresh print for each of the nine taker-action pairs (rows PM action, columns Kalshi action; `all` pools both sides), plus the two tails — the share of prints with PM − K ≤ −2 c and ≥ +2 c — and the print count. The sign is not decoration: for the four pure pairs the signed gap **is** the pre-fee gross edge of one route in each direction (`1 − p_YES − p_NO` with the legs the route reads):

| pair (PM / K) | reads | PM − K < 0 means | PM − K > 0 means |
|---|---|---|---|
| buy / sell | PM ask-side print vs K bid-side print | taker/taker, PM YES + K NO (edge = −gap) | ordinary spread crossing, no route |
| sell / buy | PM bid-side vs K ask-side | ordinary spread crossing, no route | taker/taker, K YES + PM NO (edge = +gap) |
| buy / buy | both ask-side | taker/maker, PM YES + K NO (−gap) | maker/taker, K YES + PM NO (+gap) |
| sell / sell | both bid-side | maker/taker, PM YES + K NO (−gap) | taker/maker, K YES + PM NO (+gap) |

So a cell's left tail (PM − K ≤ −2 c, buckets `(−3,−2]` and below) is the share of prints on which the "PM YES" route of that pair had at least 2 c gross, the right tail (≥ +2 c) the same for "K YES". `|PM − K|` would merge the two and lose the direction; it stays useful only as a disagreement measure (§10). Colour is the signed mean (blue = PM prints lower, red = higher); the pooled offset of about −0.4 c comes from Kalshi's buy-heavy flow printing at the ask.

In [ ]:
ORDER = ['buy', 'sell', 'all']
fig, axes = grid(1, len(CANDS), w=4.4, h=3.8)
mats = {}
for j, c in enumerate(CANDS):
    z = np.full((3, 3), np.nan); lo = np.zeros((3, 3)); hi = np.zeros((3, 3)); nn = np.zeros((3, 3))
    for (a, b) in product(range(3), repeat=2):
        d = cell(cents, c); d = d[d.pair == f'{ORDER[a]}/{ORDER[b]}']
        if len(d):
            pct, st = cent_stats(d)
            z[a, b], nn[a, b] = st['mean_gap_c'], st['prints']
            lo[a, b], hi[a, b] = pct.n[np.array(C) <= -2].sum(), pct.n[np.array(C) >= 2].sum()
    mats[j] = (z, lo, hi, nn)
vmax = max(np.nanmax(np.abs(z)) for z, *_ in mats.values())
for j, (z, lo, hi, nn) in mats.items():
    ax = axes[0, j]
    ax.imshow(z, cmap='RdBu_r', vmin=-vmax, vmax=vmax); ax.grid(False)
    for (a, b) in product(range(3), repeat=2):
        if np.isfinite(z[a, b]):
            ax.text(b, a, f'{z[a, b]:+.2f}c' + chr(10) + f'{lo[a, b]:.0f}% | {hi[a, b]:.0f}%' + chr(10) + f'{nn[a, b] / 1000:.0f}k', ha='center', va='center', fontsize=7)
    ax.set_xticks(range(3)); ax.set_xticklabels(ORDER, fontsize=8); ax.set_yticks(range(3)); ax.set_yticklabels(ORDER, fontsize=8)
    ax.set_xlabel('Kalshi taker action')
finish(fig, axes, [''], [LABEL[c] for c in CANDS], 'PM taker action', 'By action pair: signed mean PM − K | % ≤ −2c | % ≥ +2c | prints (k)')

## 5. Which outcome, which direction?

02 §4 per candidate cell, signed: the mean pooled PM − K per fresh print by outcome, and the two tails (share of fresh prints with PM − K ≤ −2 c, i.e. PM printing at least 2 c lower, and ≥ +2 c). A systematically negative draw, say, tells which venue to buy the draw on; the tails say how often the disagreement is large enough to matter after costs.

In [ ]:
OUTS = ['home', 'draw', 'away']
rows = []
for c in CANDS:
    for o in OUTS:
        d = cell(cents, c); d = d[(d.pair == 'all/all') & (d.outcome == o)]
        if len(d):
            pct, st = cent_stats(d)
            rows.append(dict(cell=LABEL[c].replace(chr(10), ' | '), outcome=o, le_m2c_pct=pct.n[np.array(C) <= -2].sum(), ge_p2c_pct=pct.n[np.array(C) >= 2].sum(), **st))
raw = pd.DataFrame(rows)
fig, axes = grid(1, len(CANDS), w=4.4, h=3.2, sharey=True)
for j, c in enumerate(CANDS):
    d = raw[raw.cell == LABEL[c].replace(chr(10), ' | ')].set_index('outcome').reindex(OUTS)
    x = np.arange(len(OUTS)); ax = axes[0, j]
    ax.bar(x - .2, d.mean_gap_c, .4, color=[BLUE, GREEN, RED], label='mean PM − K (c)')
    ax2 = ax.twinx(); ax2.grid(False)
    ax2.bar(x + .2, -d.le_m2c_pct, .2, color='grey', alpha=.6, label='% ≤ −2c (down)'); ax2.bar(x + .4, d.ge_p2c_pct, .2, color='grey', alpha=.3, label='% ≥ +2c (up)')
    ax.axhline(0, color='k', lw=.6); ax.set_xticks(x); ax.set_xticklabels(OUTS)
    if j == len(CANDS) - 1: ax2.set_ylabel('% of fresh prints beyond ±2c')
    if j == 0: ax2.legend(fontsize=7, loc='lower right')
finish(fig, axes, [''], [LABEL[c] for c in CANDS], 'mean PM − K per print (cents)', 'Signed mean gap (colour bars, left axis) and ±2c tails (grey, right axis) by outcome')
display(raw.set_index(['cell', 'outcome'])[['prints', 'mean_gap_c', 'mean_abs_gap_c', 'le_m2c_pct', 'ge_p2c_pct']].round(2))

## 6. Fee-adjusted last-print screens

At every route-leg print with the other leg ≤ F old, price both legs using their latest trades under `fees_K_maker_charged`. All four role combinations are included; route names list the PM role first and the Kalshi role second. Bars show the fraction of fresh observations with positive fee-adjusted reference edge, by route and direction.

Then compare fee scenarios at strict 5s and sweep freshness from 1 to 300s. These are last-print comparisons; future repricing diagnostics are in [02.2](02.2_cross_venue_confirmation_and_response.ipynb).

**Maker/maker: two passive-fill price references.** For `PM YES + K NO`, use the PM taker-sell YES print as the PM maker-buy YES reference, and one minus the Kalshi taker-buy YES print as the Kalshi maker-buy NO reference. The opposite direction reverses these actions. The reference net edge is `1 − PM cost − Kalshi cost − scenario fees`, with maker rebates included only in the explicitly labeled rebate scenario. Fee scenarios are model assumptions, not reconstructed historical billing.

A positive maker/maker reference edge assumes **both passive legs fill at those prices**. It does not measure joint fill probability, queue priority or the risk that only one leg fills. The added route appears in this section's main screen, fee-scenario comparison and freshness sweep; §7 also includes all four routes; §8–9 retain their existing three-route scope.

In [ ]:
scr = export('screen')
scr = scr[scr.combo.isin({c[0] for c in CANDS})]
ROUTES = ['taker/taker', 'taker/maker', 'maker/taker']
SCREEN_ROUTES = ROUTES + ['maker/maker']
DIRS = ['PM YES + K NO', 'K YES + PM NO']
def pipe_cell(frame, c):
    return frame[(frame.combo == c[0]) & (frame.phase == c[1]) & (frame.freshness == F_PIPE[c])]
def screen_rates(frame, by):
    g = frame.groupby(by, observed=True)[['n', 'n_lot', 'pos_0c', 'pos_0c_lot', 'pos_1c',
        'fresh_time', 'open_time_0c', 'edge_sum']].sum()
    g['fixtures'] = frame.groupby(by, observed=True).match_id.nunique()
    g['pos_pct'] = 100 * g.pos_0c / g.n.replace(0, np.nan)
    g['open_pct_of_fresh'] = 100 * g.open_time_0c / g.fresh_time.replace(0, np.nan)
    g['mean_net_c'] = 100 * g.edge_sum / g.n.replace(0, np.nan)
    return g.reset_index()
screen_cells = pd.concat([pipe_cell(scr, c).assign(cell=LABEL[c].replace(chr(10), ' | ')) for c in CANDS], ignore_index=True)
CELLS = [LABEL[c].replace(chr(10), ' | ') for c in CANDS]
rates = []
fig, axes = grid(len(OUTCOME_SEL), len(CANDS), sharex=True, sharey=True)
for i, (name, outcomes) in enumerate(OUTCOME_SEL.items()):
    r = screen_rates(screen_cells[screen_cells.outcome.isin(outcomes)], ['cell', 'route', 'direction']).assign(outcome_sel=name)
    rates.append(r)
    for j, cl in enumerate(CELLS):
        d = r[r.cell == cl]
        piv = lambda col: {dr: d[d.direction == dr].set_index('route').reindex(SCREEN_ROUTES)[col].values for dr in DIRS}
        bars(axes[i, j], SCREEN_ROUTES, piv('pos_pct'), [BLUE, RED])
        axes[i, j].set_title(f'pipeline F={F_PIPE[CANDS[j]]}s   n={int(d.n.sum()):,}', fontsize=8)
axes[0, 0].legend(fontsize=8, title='positive vs last print', title_fontsize=7)
finish(fig, axes, list(OUTCOME_SEL), [LABEL[c] for c in CANDS], '% of observations', 'Positive after charged fees, by route and direction')
rates = pd.concat(rates, ignore_index=True)
route_summary = rates[rates.route.isin(['taker/taker', 'maker/maker'])]
for col in ['n', 'pos_pct', 'mean_net_c', 'open_pct_of_fresh']:
    print(f'Taker/taker and maker/maker reference screen: {col}')
    display(route_summary.pivot_table(index=['cell', 'route'], columns=['outcome_sel', 'direction'], values=col).reindex(columns=['all', 'team_win', 'draw'], level=0).round(2))
# Sections 8–9 retain their existing three-route population.
scr_cells = screen_cells[screen_cells.route.isin(ROUTES)].copy()


In [ ]:
# Fee scenarios: strict 5 s and 100 contracts on both legs only (the export has no other population).
sc = export('scenarios')
sc = pd.concat([sc[(sc.combo == c[0]) & (sc.phase == c[1])].assign(cell=LABEL[c].replace(chr(10), ' | ')) for c in CANDS], ignore_index=True)
fs = sc.groupby(['cell', 'route', 'scenario'], observed=True)[['n', 'positive', 'positive_1c']].sum()
fs['positive_pct'] = 100 * fs.positive / fs.n
fs = fs.reset_index()
SCEN = list(gaps.SCENARIOS)
fig, axes = grid(1, len(CANDS), w=4.6, h=3.2, sharey=True)
for j, cl in enumerate(CELLS):
    d = fs[fs.cell == cl].pivot(index='route', columns='scenario', values='positive_pct').reindex(index=SCREEN_ROUTES, columns=SCEN)
    bars(axes[0, j], SCREEN_ROUTES, {s: d[s].values for s in SCEN}, plt.cm.tab10.colors[:len(SCEN)])
    axes[0, j].set_title(f'F={STRICT}s (strict export)   n={int(fs[fs.cell == cl].n.sum() / len(SCEN)):,}', fontsize=8)
axes[0, -1].legend(fontsize=6, loc='upper left')
finish(fig, axes, [''], [LABEL[c] for c in CANDS], '% positive (vs last print)', 'Fee / rebate scenarios, strict 5 s, 100-contract legs')
display(fs.pivot_table(index=['cell', 'route'], columns='scenario', values='positive_pct').reindex(columns=SCEN).round(1))

In [ ]:
# Last-print positive rates across freshness thresholds.
fig, axes = grid(1, len(CANDS), w=4.6, h=3.2, sharey=True)
for j, c in enumerate(CANDS):
    sw = screen_rates(scr[(scr.combo == c[0]) & (scr.phase == c[1])], ['route', 'freshness']).sort_values('freshness')
    for r_, color in zip(SCREEN_ROUTES, [BLUE, GREEN, RED, PURPLE]):
        d = sw[sw.route == r_]
        axes[0, j].plot(d.freshness, d.pos_pct, color=color, ls='--', marker='o', ms=3, label=f'{r_} vs last print')
    axes[0, j].set_xscale('log'); axes[0, j].axvline(c[2], color='grey', lw=.8, ls=':'); axes[0, j].set_xlabel('Freshness F (s)')
axes[0, 0].legend(fontsize=6)
finish(fig, axes, [''], [LABEL[c] for c in CANDS], '% of observations', "Last-print positive rates by freshness (dotted line = the cell's F)")

## 7. Positive reference edge by trigger venue and price level

Two separate conditional comparisons use the `conditions` export at **strict F=5s**, including maker/maker. Outcomes and route directions are pooled within each bucket. Each rate is positive last-print observations divided by all observations **in that same bucket and route**; it is not the share of all positives contributed by that bucket. Tables retain the observation counts so small groups remain visible.

The 60s club-market candidates are represented here only by their 5s-fresh subset. Maker/maker uses two passive-fill price references, as in §6, and does not measure joint fill probability.

In [ ]:
cd = export('conditions')
cd = pd.concat([cd[(cd.combo == c[0]) & (cd.phase == c[1])].assign(cell=LABEL[c].replace(chr(10), ' | ')) for c in CANDS], ignore_index=True)
CONDITION_ROUTES = SCREEN_ROUTES

def plot_conditions(key, levels, title):
    g = cd.groupby(['cell', 'route', key], observed=True)[['n', 'pos_0c']].sum().reset_index()
    g['pos_pct'] = 100 * g.pos_0c / g.n.replace(0, np.nan)
    fig, axes = grid(1, len(CANDS), w=4.6, h=3.2, sharey=True)
    for j, cl in enumerate(CELLS):
        d = g[g.cell == cl]
        series = {r: d[d.route == r].set_index(key).reindex(levels).pos_pct.values for r in CONDITION_ROUTES}
        bars(axes[0, j], levels, series, [BLUE, GREEN, RED, PURPLE])
        axes[0, j].set_title(f'F={STRICT}s; route-observations={int(d.n.sum()):,}', fontsize=8)
    axes[0, 0].legend(fontsize=7)
    finish(fig, axes, [''], [LABEL[c] for c in CANDS], '% positive within bucket and route', title)
    display(g.set_index(['cell', 'route', key])[['n', 'pos_0c', 'pos_pct']].round(2))


### 7a. Trigger venue

Which venue printed at the observation: `pm`, `k`, or `both` (both route legs printed in the same second). This identifies the latest update, not which venue caused a price move.

In [ ]:
plot_conditions('trigger', ['pm', 'k', 'both'], 'Positive reference edge by trigger venue — strict 5s')

### 7b. Price level

Buckets refer to the YES acquisition-price reference on the route's YES-holding venue (PM for `PM YES + K NO`, Kalshi for the reverse direction). Compare the positive-edge rate across low, middle and high price ranges, separately for each route.

In [ ]:
plot_conditions('price_bucket', list(gaps.PRICE_BUCKETS), 'Positive reference edge by YES price level — strict 5s')

## 8. How frequent? Per fixture-hour, per fixture, share of the clock

02 §7 per candidate cell at its F, `maker/maker` excluded:

1. **per fixture-hour** — positive observations per hour of the cell's phase (phase length × fixtures as denominator);
2. **per fixture** — how many fixtures show at least one / 5+ / 20+ last-print positives, and how concentrated the total is;
3. **share of the clock** — time-weighted: seconds with a fresh, positive last-print edge over the seconds with any fresh pair, and over the whole clock (six clocks per fixture: three outcomes × two directions).

In [ ]:
ph = screen_rates(scr_cells, ['cell', 'route'])
ph['combo'] = ph.cell.map({LABEL[c].replace(chr(10), ' | '): c[0] for c in CANDS}); ph['phase'] = ph.cell.map({LABEL[c].replace(chr(10), ' | '): c[1] for c in CANDS})
ph['fixture_hours'] = ph.combo.map(FIXTURES).astype(float) * ph.phase.map(PHASE_SEC) / 3600
ph['pos_per_fixture_hour'] = ph.pos_0c / ph.fixture_hours
ph['open_pct_of_clock'] = 100 * ph.open_time_0c / (6 * ph.combo.map(FIXTURES).astype(float) * ph.phase.map(PHASE_SEC))
fig, axes = grid(1, len(CANDS), w=4.4, h=3.2, sharey=True)
for j, cl in enumerate(CELLS):
    d = ph[ph.cell == cl].set_index('route').reindex(ROUTES)
    bars(axes[0, j], ROUTES, {'positive vs last print': d.pos_per_fixture_hour.values}, [BLUE])
    axes[0, j].set_yscale('log')
axes[0, 0].legend(fontsize=7)
finish(fig, axes, [''], [LABEL[c] for c in CANDS], 'per fixture-hour (log)', 'Positive observations per fixture-hour')
display(ph.set_index(['cell', 'route'])[['n', 'pos_per_fixture_hour', 'open_pct_of_fresh', 'open_pct_of_clock']].round(2))

per_fx = scr_cells.groupby(['cell', 'match_id', 'route'], observed=True).pos_0c.sum().reset_index()
dist = []
for c in CANDS:
    cl = LABEL[c].replace(chr(10), ' | '); ids = m[m.combo == c[0]].match_id
    for r_ in ROUTES:
        s = per_fx[(per_fx.cell == cl) & (per_fx.route == r_)].set_index('match_id').pos_0c.reindex(ids, fill_value=0)
        dist.append(dict(cell=cl, route=r_, fixtures=len(s), with_any=int((s > 0).sum()), with_5_plus=int((s >= 5).sum()),
                         with_20_plus=int((s >= 20).sum()), median=float(s.median()), p90=float(s.quantile(.9)),
                         top10_share_pct=100 * s.nlargest(10).sum() / max(s.sum(), 1)))
display(pd.DataFrame(dist).set_index(['cell', 'route']).round(1))

## 9. How long do they last? Event-time episodes

An episode is a maximal run of consecutive route-leg prints whose fee-adjusted edge stays positive. `span_reprice` measures seconds until a leg prints through that edge; it is a trade-tape upper bound on survival, not a fill window. This section retains episode counts, single-print share, time-to-reprice distributions and censoring, with fresh-opening evaluated at each candidate's F.

The last-positive-print lower-bound diagnostic is in [02.2 §5](02.2_cross_venue_confirmation_and_response.ipynb).

In [ ]:
ep = export('episodes', ['route', 'direction', 'outcome', 'threshold_c', 'open_sec', 'n_prints', 'span_reprice', 'censored', 'open_age', 'phase', 'match_id'])
ep = ep[(ep.threshold_c == 0) & (ep.route != 'maker/maker')]
ep = pd.concat([ep[(ep.combo == c[0]) & (ep.phase == c[1]) & (ep.open_age <= c[2])].assign(cell=LABEL[c].replace(chr(10), ' | ')) for c in CANDS], ignore_index=True)
q90 = lambda x: x.quantile(.9)
dur = ep.groupby(['cell', 'route'], observed=True).agg(
    episodes=('open_sec', 'size'), fixtures=('match_id', 'nunique'), single_print_pct=('n_prints', lambda x: 100 * (x == 1).mean()),
    median_reprice_s=('span_reprice', 'median'), p90_reprice_s=('span_reprice', q90), censored_pct=('censored', lambda x: 100 * x.mean())).round(1)
display(dur)
qs = np.linspace(0.005, 0.995, 199)
fig, axes = grid(1, len(CANDS), w=4.6, h=3.2, sharey=True)
for j, cl in enumerate(CELLS):
    for r_, color in zip(ROUTES, [BLUE, GREEN, RED]):
        x = ep[(ep.cell == cl) & (ep.route == r_)].span_reprice.dropna().to_numpy()
        if len(x):
            axes[0, j].plot(np.quantile(x, qs), qs, color=color, label=f'{r_} (n={len(x):,})')
    axes[0, j].set_xscale('log'); axes[0, j].set_xlabel('seconds until a leg printed through the edge'); axes[0, j].legend(fontsize=7)
finish(fig, axes, [''], [LABEL[c] for c in CANDS], 'ECDF', 'Episodes: seconds to reprice (fresh-opening at the cell F, threshold 0)')

## Summary

- Full-clock activity and source-trade waiting use different denominators. Keep the fresh, stale and no-prior-print trigger groups separate when interpreting §2.
- The 70% rule selects World Cup in play (F=5s), World Cup's final pregame hour (F=10s), and Premier League / Champions League 2025/26 in play (F=60s). This scopes the fresh-pair gap screen; it does not rule out asynchronous strategies elsewhere.
- §3–8 describe last-print reference prices, fee-adjusted screens and their frequency. Changes with F can reflect stale references; a positive screen is not an executable opportunity.
- §9 retains event-time episode durations. The next stage is [02.2](02.2_cross_venue_confirmation_and_response.ipynb), which examines subsequent prices, passive-fill diagnostics and baskets.